**Interpretación de Negocio: Ingesta de Datos**
Para iniciar el análisis, establecimos un flujo de ingesta de datos automatizado utilizando Pandas para la lectura estructurada del archivo CSV y SQLite3 como motor de base de datos local. La información de los empleados se almacenó con éxito en la tabla `Detalle` dentro de la base de datos `RH.db`, estandarizando los tipos de datos y dejando el terreno preparado para la extracción de métricas mediante lenguaje SQL.

In [4]:
import pandas as pd
import sqlite3

# 1. Leer el archivo CSV
df_rh = pd.read_csv('recursos_humanos.csv')

# 2. Conectar a la base de datos SQLite
conexion = sqlite3.connect('RH.db')

# 3. Exportar el DataFrame a la tabla 'Detalle' en SQLite
# Usamos if_exists='replace' 
df_rh.to_sql(name='Detalle', con=conexion, if_exists='replace', index=False)

# 4. Consulta de prueba rápida para verificar que se cargó bien
query_prueba = "SELECT * FROM Detalle LIMIT 5;"
df_prueba = pd.read_sql_query(query_prueba, conexion)

display(df_prueba)

,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years,sales,salary
0,0.38,0.53,2,157,3,0,1,0,sales,low
1,0.80,0.86,5,262,6,0,1,0,sales,medium
2,0.11,0.88,7,272,4,0,1,0,sales,medium
3,0.72,0.87,5,223,5,0,1,0,sales,low
4,0.37,0.52,2,159,3,0,1,0,sales,low


**Interpretación de Negocio: Análisis de Satisfacción**
Mediante la instrucción `ORDER BY DESC`, hemos priorizado la visualización de los empleados con el nivel de satisfacción más alto en la cima del reporte. Al cruzar esta métrica con su departamento y nivel salarial, el equipo de Recursos Humanos puede identificar rápidamente el perfil de los colaboradores más conformes y buscar patrones (por ejemplo, si los salarios altos o ciertos departamentos concentran a los empleados más felices) para replicar esas condiciones en áreas de menor rendimiento.

In [5]:
query_1 = """
SELECT sales, salary, satisfaction_level
FROM Detalle
ORDER BY satisfaction_level DESC;
"""
df_paso1 = pd.read_sql_query(query_1, conexion)
display(df_paso1.head(10))

,sales,salary,satisfaction_level
0,technical,low,1.0
1,technical,low,1.0
2,support,low,1.0
3,sales,low,1.0
4,technical,low,1.0
5,marketing,high,1.0
6,hr,low,1.0
7,support,low,1.0
8,technical,low,1.0
9,sales,medium,1.0


**Interpretación de Negocio: Rendimiento por Departamento**
A través de la instrucción `GROUP BY` combinada con la función de agregación `AVG()`, logramos consolidar el desempeño a nivel macro. Este resultado permite a la dirección identificar rápidamente si existen asimetrías de rendimiento entre los distintos departamentos. Por ejemplo, si un área muestra un promedio de evaluación significativamente más bajo, podría ser un indicador temprano de problemas de liderazgo, falta de capacitación o procesos ineficientes en ese sector específico, permitiendo dirigir los recursos de mejora de forma estratégica.

In [6]:
query_2 = """
SELECT sales, AVG(last_evaluation) AS promedio_evaluacion
FROM Detalle
GROUP BY sales;
"""
df_paso2 = pd.read_sql_query(query_2, conexion)
display(df_paso2)

,sales,promedio_evaluacion
0,IT,0.716830
1,RandD,0.712122
2,accounting,0.717718
3,hr,0.708850
4,management,0.724000
5,marketing,0.715886
6,product_mng,0.714756
7,sales,0.709717
8,support,0.723109
9,technical,0.721099


**Interpretación de Negocio: Tasa de Riesgo Laboral (Alertas)**
Aquí aplicamos un nivel más profundo de análisis utilizando la cláusula `HAVING`. A diferencia de `WHERE` (que filtra fila por fila antes de agrupar), `HAVING` nos permite filtrar los grupos ya creados basándonos en una condición agregada. En términos de negocio, esto funciona como un sistema de alertas: le pedimos al motor de base de datos que descarte el ruido general y solo nos exponga aquellos departamentos que superan un umbral crítico de riesgo (un promedio de accidentes mayor al 15%). Esto entrega información accionable e inmediata al equipo de Salud y Seguridad Ocupacional, indicándoles exactamente en qué áreas deben priorizar las auditorías preventivas.

In [7]:
query_3 = """
SELECT sales, AVG(Work_accident) AS promedio_accidentes
FROM Detalle
GROUP BY sales
HAVING AVG(Work_accident) > 0.15;
"""
df_paso3 = pd.read_sql_query(query_3, conexion)
display(df_paso3)

,sales,promedio_accidentes
0,RandD,0.170267
1,management,0.163492
2,marketing,0.160839
3,support,0.154778


**Interpretación de Negocio: Volumen Total de Incidentes**
Para responder a esta nueva interrogante, la consulta cambia fundamentalmente en su métrica de agregación: pasamos de utilizar promedios (`AVG`), que miden el riesgo relativo o tasa de incidencia, a utilizar sumas (`SUM`), que miden el volumen absoluto de incidentes. Al filtrar con `HAVING SUM(Work_accident) > 200`, el enfoque del negocio cambia. Mientras que la consulta anterior revelaba departamentos pequeños con alta probabilidad de riesgo, esta nueva perspectiva nos expone aquellos departamentos (probablemente los más grandes) que aportan la mayor carga volumétrica de accidentes a la empresa. Ambas métricas son complementarias para diseñar una estrategia integral de seguridad corporativa.

In [8]:
query_4 = """
SELECT sales, SUM(Work_accident) AS total_accidentes
FROM Detalle
GROUP BY sales
HAVING SUM(Work_accident) > 200;
"""
df_paso4 = pd.read_sql_query(query_4, conexion)
display(df_paso4)

,sales,total_accidentes
0,sales,587
1,support,345
2,technical,381
